### Preparation

In [1]:
from gitsource import GithubRepositoryDataReader

github_reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

fetched_files = github_reader.read()

parsed_documents = []
for file in fetched_files:
    parsed_doc = file.parse()
    parsed_documents.append(parsed_doc)

### Q1. How many lesson pages

In [2]:
total_pages = len(parsed_documents)
print(f"The total number of processed lesson pages is {total_pages}.")

The total number of processed lesson pages is 72.


### Q2. Indexing and searching

In [3]:
from minsearch import Index

document_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

document_index.fit(parsed_documents)

search_query = "How does the agentic loop keep calling the model until it stops?"
search_results = document_index.search(search_query)

top_search_result = search_results[0]
top_filename = top_search_result["filename"]

print(f"The most relevant document filename (top 1) is '{top_filename}'.")

The most relevant document filename (top 1) is '01-agentic-rag/lessons/14-agentic-loop.md'.


### Q3. RAG

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from rag_helper import RAGCustom
from IPython.display import display, Markdown

load_dotenv()

openai_client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

rag_assistant = RAGCustom( 
    index=document_index,
    llm_client=openai_client,
    model="openai/gpt-oss-120b:free",
)

rag_response = rag_assistant.rag(search_query)
display(Markdown(rag_response))

The agentic loop is just a **`while True`** that repeats the same three steps over and over:

1. **Call the model** with the current message history (`messages`).  
2. **Inspect the model’s output**:  
   * If the output contains a `function_call`, the loop runs the requested tool (via `make_call`) and appends the tool’s result to `messages`.  
   * If the output contains a normal `message`, it is printed (or stored) as the assistant’s answer.  
3. **Set a flag** (`has_function_calls`) that records whether any function calls were seen in this turn.  

After processing all items in the response the loop increments an iteration counter and then checks the flag:

```python
if has_function_calls == False:
    break          # no function calls → we are done
```

Because the flag is **reset to `False` at the start of each iteration**, the loop will keep sending the updated `messages` back to the model **as long as the model continues to request tool calls**. When a turn returns only ordinary messages (i.e., no `function_call` entries), `has_function_calls` stays `False` and the `break` statement exits the loop.  

Thus the loop “keeps calling the model until it stops” by:

* repeatedly feeding the full conversation (including tool results) back to the LLM, and  
* terminating only when the LLM produces a response with **no function calls**, indicating it is ready to give the final answer.

In [5]:
input_token_count = rag_assistant.rag_input_tokens(search_query)

print(f"The total number of input tokens used before chunking is {input_token_count}.")

The total number of input tokens used before chunking is 7202.


### Q4. Chunking

In [6]:
from gitsource import chunk_documents

document_chunks = chunk_documents(parsed_documents, size=2000, step=1000)

print(f"The total number of generated chunks is {len(document_chunks)}.")

The total number of generated chunks is 295.


### Q5. RAG with chunking

In [7]:
chunked_document_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunked_document_index.fit(document_chunks)

chunked_rag_assistant = RAGCustom(
    index=chunked_document_index,
    llm_client=openai_client,
    model="openai/gpt-oss-120b:free",
)

display(Markdown(chunked_rag_assistant.rag(search_query)))

The **agentic loop** is just a `while‑True` that keeps sending the current
message history to the LLM, runs any tool calls the model returns, and then
feeds the tool results back into the history.  

```python
it = 1
while True:
    has_function_calls = False                # ← reset flag

    response = openai_client.responses.create(
        model=…,
        input=messages,
        tools=[search_tool]
    )
    messages.extend(response.output)          # add model output to memory

    for item in response.output:              # look at every piece the model sent back
        if item.type == "function_call":       # ‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑‑-
            call_output = make_call(item)     # run the tool
            messages.append(call_output)      # add tool result to memory
            has_function_calls = True         # we did a call this turn
        elif item.type == "message":
            # just a normal assistant reply – print / store it
            …

    it += 1

    # ---- exit condition -------------------------------------------------
    # If the model did **not** emit any function calls on this turn,
    # `has_function_calls` stays False and we break out of the loop.
    if not has_function_calls:
        break
```

### How it stops
1. **Flag reset** – At the start of each iteration `has_function_calls` is set to
   `False`.
2. **Detect calls** – While iterating over `response.output`, if an item’s
   `type` is `"function_call"` the code runs the corresponding tool and sets
   `has_function_calls = True`.
3. **Check after the turn** – After processing all items, the loop tests the flag:
   *If `has_function_calls` is still `False` (i.e., the model returned only normal
   messages and no tool calls), the `break` statement fire and the loop ends.*

So the loop **keeps calling the model** as long as the model keeps asking for a
tool. The moment the model gives a final answer **without any function calls**, the
flag remains `False` and the loop exits.  

(Frameworks often add extra safety nets—max‑iteration count, token budget,
time limit—but the core stopping logic is exactly this `has_function_calls`
flag.)

In [8]:
chunked_input_token_count = chunked_rag_assistant.rag_input_tokens(search_query)

print(f"The total number of input tokens used after chunking the documents is {chunked_input_token_count}.")

The total number of input tokens used after chunking the documents is 2414.


### Q6. Turning it into an agent

In [9]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search_faq_database(self, query, num_results=5):
    """
    Search the FAQ database for entries matching the given query.
    """
    return chunked_document_index.search(
        query,
        num_results=num_results,
    )

agent_tools = Tools()
agent_tools.add_tool(search_faq_database)

In [10]:
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface)

system_instruction = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

agent_runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=system_instruction,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model="openai/gpt-oss-120b:free",
        client=openai_client
    ),
)

In [11]:
agent_runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=runner_callback
)

print("\nThe agentic loop successfully ran using the Tools (search_faq_database) and responded to the question in the conversation format above.")

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


Aspect,Plain RAG (retrieval‑augmented generation),Agentic loop (LLM‑driven tool‑use)
Control flow,"Fixed, three‑step pipeline that you write yourself: 1️⃣ search → 2️⃣ build prompt → 3️⃣ LLM generation.All steps run once and in that exact order.","A while‑loop that the LLM controls. The model can: • Call a tool (search, browse, compute, etc.) • Receive the tool’s output • Decide whether to call another tool or produce a final answer • Repeat until it says “Done”. The number and order of steps are dynamic."
Decision making,You decide when to call the retriever and how to format its results. The LLM never decides to run a tool; it only sees whatever you give it.,"The LLM itself decides whether a tool call is needed, which tool, and when to stop. This is the “agentic” part – the model acts as a planner/executor."
Tool variety,Usually only a single retrieval component (keyword or vector search). The rest of the pipeline is pure prompting.,"Any number of function‑calling tools can be registered: web search, database query, code execution, file I/O, etc. The loop can switch between them arbitrarily."
Looping / multi‑turn,"One‑shot: you run the pipeline, get a response, and you’re done. If the answer is incomplete you have to start a new RAG call yourself.","Built‑in multi‑turn management: the loop automatically keeps the full message history (all_messages) and token usage, letting the model refine its answer over several iterations."
Cost visibility,You only see the cost of the single LLM call (plus the cost of the search if you bill it).,"The framework reports per‑iteration usage and a cumulative cost (result.cost). Because the model may make many calls, you can monitor how tool usage adds up."
Implementation complexity,Very simple to code: a function def rag(q): … that calls a retriever and then the LLM.,"Slightly more complex, but many libraries/frameworks give you a ready‑made AgentRunner or AIAgent that hides the loop logic. You just define the goal, the tool list, and optionally a system prompt."
When to use,• You know the exact retrieval‑and‑generation steps. • Latency and cost must be minimal. • The problem is purely “fetch‑and‑answer”.,"• The workflow is undetermined ahead of time (e.g., need to decide whether to search the web, call a calculator, or write code). • You want the model to self‑orchestrate tools and iterate until a satisfactory answer is produced."
Typical output,A single answer that may be hallucinated if the retrieved context is insufficient.,"A series of messages showing tool calls, their results, and a final answer that is grounded in those intermediate results."



The agentic loop successfully ran using the Tools (search_faq_database) and responded to the question in the conversation format above.


/home/gustav/Gudang/LLM-Zoomcamp-Homeworks/01-agentic-rag/.venv/lib/python3.14/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model 'free'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(
